In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

session = get_active_session()

In [ ]:
N_ROWS = 1000

poi_mock_raw = session.sql(f"""
    WITH lookup_numbered AS (
        SELECT
            code,
            fclass,
            ROW_NUMBER() OVER (ORDER BY code) - 1 AS lookup_idx
        FROM VALUES
            (2001,'police'),(2002,'fire_station'),(2004,'post_box'),
            (2005,'post_office'),(2006,'telephone'),(2007,'library'),
            (2008,'town_hall'),(2009,'courthouse'),(2010,'prison'),
            (2011,'embassy'),(2012,'community_centre'),(2013,'nursing_home'),
            (2014,'arts_centre'),(2015,'graveyard'),(2016,'market_place'),
            (2017,'consulate'),(2030,'recycling'),(2031,'recycling_glass'),
            (2032,'recycling_paper'),(2033,'recycling_clothes'),(2034,'recycling_metal'),
            (2081,'university'),(2082,'school'),(2083,'kindergarten'),
            (2084,'college'),(2099,'public_building'),(2101,'pharmacy'),
            (2110,'hospital'),(2111,'clinic'),(2120,'doctors'),
            (2121,'dentist'),(2129,'veterinary'),(2201,'theatre'),
            (2202,'nightclub'),(2203,'cinema'),(2204,'park'),
            (2205,'playground'),(2206,'dog_park'),(2251,'sports_centre'),
            (2252,'pitch'),(2253,'swimming_pool'),(2254,'tennis_court'),
            (2255,'golf_course'),(2256,'stadium'),(2257,'ice_rink'),
            (2301,'restaurant'),(2302,'fast_food'),(2303,'cafe'),
            (2304,'pub'),(2305,'bar'),(2306,'food_court'),(2307,'biergarten'),
            (2401,'hotel'),(2402,'motel'),(2403,'bed_and_breakfast'),
            (2404,'guesthouse'),(2405,'hostel'),(2406,'chalet'),
            (2421,'shelter'),(2422,'camp_site'),(2423,'alpine_hut'),
            (2424,'caravan_site'),(2501,'supermarket'),(2502,'bakery'),
            (2503,'kiosk'),(2504,'mall'),(2505,'department_store'),
            (2510,'general'),(2511,'convenience'),(2512,'clothes'),
            (2513,'florist'),(2514,'chemist'),(2515,'bookshop'),(2516,'butcher'),
            (2517,'shoe_shop'),(2518,'beverages'),(2519,'optician'),
            (2520,'jeweller'),(2521,'gift_shop'),(2522,'sports_shop'),
            (2523,'stationery'),(2524,'outdoor_shop'),(2525,'mobile_phone_shop'),
            (2526,'toy_shop'),(2527,'newsagent'),(2528,'greengrocer'),
            (2529,'beauty_shop'),(2530,'video_shop'),(2541,'car_dealership'),
            (2542,'bicycle_shop'),(2543,'doityourself'),(2544,'furniture_shop'),
            (2546,'computer_shop'),(2547,'garden_centre'),(2561,'hairdresser'),
            (2562,'car_repair'),(2563,'car_rental'),(2564,'car_wash'),
            (2565,'car_sharing'),(2566,'bicycle_rental'),(2567,'travel_agent'),
            (2568,'laundry'),(2590,'vending_machine'),(2591,'vending_cigarette'),
            (2592,'vending_parking'),(2593,'vending_drinks'),(2594,'vending_tickets'),
            (2601,'bank'),(2602,'atm'),(2701,'tourist_info'),(2704,'tourist_map'),
            (2705,'tourist_board'),(2706,'tourist_guidepost'),(2721,'attraction'),
            (2722,'museum'),(2723,'monument'),(2724,'memorial'),(2725,'art'),
            (2731,'castle'),(2732,'ruins'),(2733,'archaeological'),
            (2734,'wayside_cross'),(2735,'wayside_shrine'),(2736,'battlefield'),
            (2737,'fort'),(2741,'picnic_site'),(2742,'viewpoint'),
            (2743,'zoo'),(2744,'theme_park'),(2901,'toilet'),(2902,'bench'),
            (2903,'drinking_water'),(2904,'fountain'),(2905,'hunting_stand'),
            (2906,'waste_basket'),(2907,'camera_surveillance'),
            (2921,'emergency_phone'),(2922,'fire_hydrant'),(2923,'emergency_access'),
            (2950,'tower'),(2951,'tower_comms'),(2952,'water_tower'),
            (2953,'tower_observation'),(2954,'windmill'),(2955,'lighthouse'),
            (2961,'wastewater_plant'),(2962,'water_well'),
            (2963,'water_mill'),(2964,'water_works')
        AS t(code, fclass)
    ),
    raw AS (
        SELECT
            ABS(RANDOM() % 9000000) + 1000000                        AS osm_id,
            ABS(RANDOM() % 149)                                      AS lookup_idx,
            ROUND(UNIFORM(-2.72::FLOAT, -2.48::FLOAT, RANDOM()), 5) AS lon,
            ROUND(UNIFORM(51.40::FLOAT, 51.55::FLOAT, RANDOM()), 5) AS lat
        FROM TABLE(GENERATOR(ROWCOUNT => {N_ROWS}))
    ),
    joined AS (
        SELECT r.osm_id, l.code, l.fclass, r.lon, r.lat
        FROM raw r
        JOIN lookup_numbered l ON r.lookup_idx = l.lookup_idx
    )
    SELECT
        osm_id,
        code,
        fclass,
        CASE
            WHEN fclass IN (
                'restaurant','cafe','pub','bar','fast_food','hotel','museum',
                'library','school','university','hospital','theatre','cinema',
                'sports_centre','attraction','castle','monument','zoo',
                'theme_park','supermarket','mall','department_store','bank',
                'park','stadium'
            )
            AND UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) > 0.4
            THEN
                CASE ABS(RANDOM() % 8)
                    WHEN 0 THEN 'North ' WHEN 1 THEN 'South '
                    WHEN 2 THEN 'East '  WHEN 3 THEN 'West '
                    WHEN 4 THEN 'Upper ' WHEN 5 THEN 'Lower '
                    WHEN 6 THEN 'Old '   ELSE        'New '
                END ||
                CASE ABS(RANDOM() % 8)
                    WHEN 0 THEN 'Bristol '    WHEN 1 THEN 'Clifton '
                    WHEN 2 THEN 'Redland '    WHEN 3 THEN 'Bedminster '
                    WHEN 4 THEN 'Bishopston ' WHEN 5 THEN 'Horfield '
                    WHEN 6 THEN 'Totterdown ' ELSE        'Easton '
                END ||
                INITCAP(REPLACE(fclass, '_', ' '))
            ELSE NULL
        END AS name,
        TO_GEOGRAPHY(
            'MULTIPOLYGON (((' ||
                lon         || ' ' || lat         || ', ' ||
                (lon+0.001) || ' ' || lat         || ', ' ||
                (lon+0.001) || ' ' || (lat+0.001) || ', ' ||
                lon         || ' ' || (lat+0.001) || ', ' ||
                lon         || ' ' || lat         ||
            ')))'
        ) AS geometry
    FROM joined
""")
print(f"✓ Mock POI data stored in poi_mock_raw ({poi_mock_raw.count()} rows)")
poi_mock_raw.show(5)

In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.functions import uniform, random, col, lit, when

poi_mock_dirty = poi_mock_raw.select(
    col("OSM_ID"),
    when(uniform(lit(0).cast("float"), lit(1).cast("float"), random()) < lit(0.015), lit(9999))
        .otherwise(col("CODE")).alias("CODE"),
    when(uniform(lit(0).cast("float"), lit(1).cast("float"), random()) < lit(0.03), lit("unknown_class"))
        .otherwise(col("FCLASS")).alias("FCLASS"),
    when(uniform(lit(0).cast("float"), lit(1).cast("float"), random()) < lit(0.02), lit(""))
        .otherwise(col("NAME")).alias("NAME"),
    col("GEOMETRY")
)
print(f"✓ Dirty poi data stored in transport_mock_dirty ({poi_mock_dirty.count()} rows)")

In [ ]:
fclass_lookup = {
    2001:'police',          2002:'fire_station',      2004:'post_box',
    2005:'post_office',     2006:'telephone',         2007:'library',
    2008:'town_hall',       2009:'courthouse',        2010:'prison',
    2011:'embassy',         2012:'community_centre',  2013:'nursing_home',
    2014:'arts_centre',     2015:'graveyard',         2016:'market_place',
    2017:'consulate',       2030:'recycling',         2031:'recycling_glass',
    2032:'recycling_paper', 2033:'recycling_clothes', 2034:'recycling_metal',
    2081:'university',      2082:'school',            2083:'kindergarten',
    2084:'college',         2099:'public_building',   2101:'pharmacy',
    2110:'hospital',        2111:'clinic',            2120:'doctors',
    2121:'dentist',         2129:'veterinary',        2201:'theatre',
    2202:'nightclub',       2203:'cinema',            2204:'park',
    2205:'playground',      2206:'dog_park',          2251:'sports_centre',
    2252:'pitch',           2253:'swimming_pool',     2254:'tennis_court',
    2255:'golf_course',     2256:'stadium',           2257:'ice_rink',
    2301:'restaurant',      2302:'fast_food',         2303:'cafe',
    2304:'pub',             2305:'bar',               2306:'food_court',
    2307:'biergarten',      2401:'hotel',             2402:'motel',
    2403:'bed_and_breakfast', 2404:'guesthouse',      2405:'hostel',
    2406:'chalet',          2421:'shelter',           2422:'camp_site',
    2423:'alpine_hut',      2424:'caravan_site',      2501:'supermarket',
    2502:'bakery',          2503:'kiosk',             2504:'mall',
    2505:'department_store', 2510:'general',          2511:'convenience',
    2512:'clothes',         2513:'florist',           2514:'chemist',
    2515:'bookshop',        2516:'butcher',           2517:'shoe_shop',
    2518:'beverages',       2519:'optician',          2520:'jeweller',
    2521:'gift_shop',       2522:'sports_shop',       2523:'stationery',
    2524:'outdoor_shop',    2525:'mobile_phone_shop', 2526:'toy_shop',
    2527:'newsagent',       2528:'greengrocer',       2529:'beauty_shop',
    2530:'video_shop',      2541:'car_dealership',    2542:'bicycle_shop',
    2543:'doityourself',    2544:'furniture_shop',    2546:'computer_shop',
    2547:'garden_centre',   2561:'hairdresser',       2562:'car_repair',
    2563:'car_rental',      2564:'car_wash',          2565:'car_sharing',
    2566:'bicycle_rental',  2567:'travel_agent',      2568:'laundry',
    2590:'vending_machine', 2591:'vending_cigarette', 2592:'vending_parking',
    2593:'vending_drinks',  2594:'vending_tickets',   2601:'bank',
    2602:'atm',             2701:'tourist_info',      2704:'tourist_map',
    2705:'tourist_board',   2706:'tourist_guidepost', 2721:'attraction',
    2722:'museum',          2723:'monument',          2724:'memorial',
    2725:'art',             2731:'castle',            2732:'ruins',
    2733:'archaeological',  2734:'wayside_cross',     2735:'wayside_shrine',
    2736:'battlefield',     2737:'fort',              2741:'picnic_site',
    2742:'viewpoint',       2743:'zoo',               2744:'theme_park',
    2901:'toilet',          2902:'bench',             2903:'drinking_water',
    2904:'fountain',        2905:'hunting_stand',     2906:'waste_basket',
    2907:'camera_surveillance', 2921:'emergency_phone', 2922:'fire_hydrant',
    2923:'emergency_access', 2950:'tower',            2951:'tower_comms',
    2952:'water_tower',     2953:'tower_observation', 2954:'windmill',
    2955:'lighthouse',      2961:'wastewater_plant',  2962:'water_well',
    2963:'water_mill',      2964:'water_works',
}

df = poi_mock_dirty.select("OSM_ID", "CODE", "FCLASS", "NAME").to_pandas()
df.columns = df.columns.str.lower()
print("Loaded shape:", df.shape)

In [ ]:
print("Null counts:")
print(df.isnull().sum())

print("\nEmpty string names:")
print((df["name"].str.strip() == "").sum())

print("\nDuplicate osm_ids")
dupes = df[df.duplicated(subset="osm_id", keep=False)]
print(f"{len(dupes)} rows with duplicate osm_id")

In [ ]:
def is_valid(row):
    if row["code"] not in fclass_lookup:
        return False
    if row["fclass"] != fclass_lookup[row["code"]]:
        return False
    if fclass_lookup.get(row["code"]) and row["code"] != {v: k for k, v in fclass_lookup.items()}.get(row["fclass"]):
        return False
    return True

before = len(df)
df = df[df.apply(is_valid, axis=1)].reset_index(drop=True)
after = len(df)

print(f"Removed {before - after} rows with code/fclass mismatches")
print(f"Remaining rows: {after}")

In [ ]:
cleaned_df = df.copy()
cleaned_df["name"] = cleaned_df["name"].str.strip().replace("", pd.NA)

# Drop duplicate osm_ids, keeping first occurrence
before_dedup = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates(subset="osm_id", keep="first").reset_index(drop=True)
after_dedup = len(cleaned_df)

print(f"Removed {before_dedup - after_dedup} duplicate osm_id rows")
print(f"Cleaned {len(cleaned_df)} rows")
print(f"Names with values: {cleaned_df['name'].notna().sum()}")
print(f"Names missing: {cleaned_df['name'].isna().sum()}")

In [ ]:
session.use_schema("SILVER")
output_table = f"silver_landuse"
cleaned_df.write.mode("overwrite").save_as_table(output_table)
print(f"Saved to AIRBNB_INVESTMENT_APP.SILVER.{output_table}")